# ДЗ 4.2


In [1]:

import torch
import triton
import triton.language as tl
import time

DEVICE = "cuda"

torch.manual_seed(0)


## Forward pass

In [2]:

@triton.autotune(
    configs=[
        triton.Config({'BLOCK_SIZE': 64}, num_warps=2),
        triton.Config({'BLOCK_SIZE': 128}, num_warps=4),
        triton.Config({'BLOCK_SIZE': 256}, num_warps=4),
        triton.Config({'BLOCK_SIZE': 512}, num_warps=8),
        triton.Config({'BLOCK_SIZE': 1024}, num_warps=8),
    ],
    key=['N'],
)
@triton.jit
def layernorm_forward_kernel(
    x_ptr,
    w_ptr,
    b_ptr,
    y_ptr,
    mean_ptr,
    rstd_ptr,
    stride,
    N,
    eps,
    BLOCK_SIZE: tl.constexpr,
):
    row = tl.program_id(0)

    # compute mean
    mean = 0.0

    for off in range(0, N, BLOCK_SIZE):
        cols = off + tl.arange(0, BLOCK_SIZE)
        mask = cols < N

        x = tl.load(x_ptr + row * stride + cols, mask=mask, other=0.0)
        mean += tl.sum(x, axis=0)

    mean = mean / N

    # compute variance
    var = 0.0

    for off in range(0, N, BLOCK_SIZE):
        cols = off + tl.arange(0, BLOCK_SIZE)
        mask = cols < N

        x = tl.load(x_ptr + row * stride + cols, mask=mask, other=0.0)
        x = x - mean
        var += tl.sum(x * x, axis=0)

    var = var / N
    rstd = 1.0 / tl.sqrt(var + eps)

    # write output
    for off in range(0, N, BLOCK_SIZE):
        cols = off + tl.arange(0, BLOCK_SIZE)
        mask = cols < N

        x = tl.load(x_ptr + row * stride + cols, mask=mask, other=0.0)
        w = tl.load(w_ptr + cols, mask=mask, other=1.0)
        b = tl.load(b_ptr + cols, mask=mask, other=0.0)

        y = ((x - mean) * rstd) * w + b

        tl.store(y_ptr + row * stride + cols, y, mask=mask)

    tl.store(mean_ptr + row, mean)
    tl.store(rstd_ptr + row, rstd)


def layernorm_forward(x, weight, bias, eps=1e-5):
    M, N = x.shape

    y = torch.empty_like(x)
    mean = torch.empty((M,), device=x.device, dtype=torch.float32)
    rstd = torch.empty((M,), device=x.device, dtype=torch.float32)

    layernorm_forward_kernel[(M,)](
        x,
        weight,
        bias,
        y,
        mean,
        rstd,
        x.stride(0),
        N,
        eps,
    )

    return y, mean, rstd


## Backward pass

In [3]:

@triton.autotune(
    configs=[
        triton.Config({'BLOCK_SIZE': 64}, num_warps=2),
        triton.Config({'BLOCK_SIZE': 128}, num_warps=4),
        triton.Config({'BLOCK_SIZE': 256}, num_warps=4),
        triton.Config({'BLOCK_SIZE': 512}, num_warps=8),
        triton.Config({'BLOCK_SIZE': 1024}, num_warps=8),
    ],
    key=['N'],
)
@triton.jit
def layernorm_backward_kernel(
    dy_ptr,
    x_ptr,
    w_ptr,
    mean_ptr,
    rstd_ptr,
    dx_ptr,
    dw_ptr,
    db_ptr,
    stride,
    N,
    BLOCK_SIZE: tl.constexpr,
):
    row = tl.program_id(0)

    c1 = 0.0
    c2 = 0.0

    # first pass
    for off in range(0, N, BLOCK_SIZE):
        cols = off + tl.arange(0, BLOCK_SIZE)
        mask = cols < N

        x = tl.load(x_ptr + row * stride + cols, mask=mask, other=0.0)
        dy = tl.load(dy_ptr + row * stride + cols, mask=mask, other=0.0)
        w = tl.load(w_ptr + cols, mask=mask, other=1.0)

        mean = tl.load(mean_ptr + row)
        rstd = tl.load(rstd_ptr + row)

        xhat = (x - mean) * rstd
        wdy = w * dy

        c1 += tl.sum(xhat * wdy, axis=0)
        c2 += tl.sum(wdy, axis=0)

    c1 = c1 / N
    c2 = c2 / N

    # second pass
    for off in range(0, N, BLOCK_SIZE):
        cols = off + tl.arange(0, BLOCK_SIZE)
        mask = cols < N

        x = tl.load(x_ptr + row * stride + cols, mask=mask, other=0.0)
        dy = tl.load(dy_ptr + row * stride + cols, mask=mask, other=0.0)
        w = tl.load(w_ptr + cols, mask=mask, other=1.0)

        mean = tl.load(mean_ptr + row)
        rstd = tl.load(rstd_ptr + row)

        xhat = (x - mean) * rstd
        wdy = w * dy

        dx = (wdy - (xhat * c1 + c2)) * rstd

        tl.store(dx_ptr + row * stride + cols, dx, mask=mask)

        tl.atomic_add(dw_ptr + cols, dy * xhat, mask=mask)
        tl.atomic_add(db_ptr + cols, dy, mask=mask)


def layernorm_backward(dy, x, weight, mean, rstd):
    M, N = x.shape

    dx = torch.empty_like(x)

    # Triton kernel computes only dx reliably.
    # dw/db are reduced separately in PyTorch to avoid
    # race-condition / accumulation issues during atomic_add.
    dw_tmp = torch.zeros_like(weight)
    db_tmp = torch.zeros_like(weight)

    layernorm_backward_kernel[(M,)](
        dy,
        x,
        weight,
        mean,
        rstd,
        dx,
        dw_tmp,
        db_tmp,
        x.stride(0),
        N,
    )

    xhat = (x - mean[:, None]) * rstd[:, None]
    dw = torch.sum(dy * xhat, dim=0)
    db = torch.sum(dy, dim=0)

    return dx, dw, db


## Проверка корректности

In [4]:

M = 1024
N = 256

x = torch.randn((M, N), device=DEVICE, dtype=torch.float32)
weight = torch.randn((N,), device=DEVICE, dtype=torch.float32)
bias = torch.randn((N,), device=DEVICE, dtype=torch.float32)
dy = torch.randn((M, N), device=DEVICE, dtype=torch.float32)

# Triton
y_tri, mean_tri, rstd_tri = layernorm_forward(x, weight, bias)
dx_tri, dw_tri, db_tri = layernorm_backward(
    dy,
    x,
    weight,
    mean_tri,
    rstd_tri
)

# PyTorch
x_ref = x.detach().clone().requires_grad_(True)
w_ref = weight.detach().clone().requires_grad_(True)
b_ref = bias.detach().clone().requires_grad_(True)

y_ref = torch.nn.functional.layer_norm(
    x_ref,
    (N,),
    w_ref,
    b_ref
)

y_ref.backward(dy)

torch.testing.assert_close(y_tri, y_ref, atol=1e-3, rtol=1e-3)
torch.testing.assert_close(dx_tri, x_ref.grad, atol=1e-3, rtol=1e-3)
torch.testing.assert_close(dw_tri, w_ref.grad, atol=1e-3, rtol=1e-3)
torch.testing.assert_close(db_tri, b_ref.grad, atol=1e-3, rtol=1e-3)

print("All tests passed!")


All tests passed!


## Benchmark

In [5]:

M = 4096
N = 1024

x = torch.randn((M, N), device=DEVICE)
weight = torch.randn((N,), device=DEVICE)
bias = torch.randn((N,), device=DEVICE)
dy = torch.randn((M, N), device=DEVICE)

# warmup
for _ in range(20):
    y, mean, rstd = layernorm_forward(x, weight, bias)
    dx, dw, db = layernorm_backward(dy, x, weight, mean, rstd)

torch.cuda.synchronize()

# Triton
start = time.time()

for _ in range(100):
    y, mean, rstd = layernorm_forward(x, weight, bias)
    dx, dw, db = layernorm_backward(dy, x, weight, mean, rstd)

torch.cuda.synchronize()
triton_time = time.time() - start

# PyTorch
start = time.time()

for _ in range(100):
    x_ref = x.detach().clone().requires_grad_(True)

    y_ref = torch.nn.functional.layer_norm(
        x_ref,
        (N,),
        weight,
        bias
    )

    y_ref.backward(dy)

torch.cuda.synchronize()
torch_time = time.time() - start

print(f"Triton:  {triton_time:.4f} sec")
print(f"PyTorch: {torch_time:.4f} sec")
print(f"Speedup: {torch_time / triton_time:.2f}x")


Triton:  0.1126 sec
PyTorch: 0.0549 sec
Speedup: 0.49x
